[![Open In Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=white)](https://colab.research.google.com/gitlab/ivanvladimir/maquinas_notebooks/blob/main/basic/common_interface_machines.ipynb)

This notebook illustrates the machines api in the _maquina_ library

In [ ]:
# Load, or install and load the library
try:
    import maquinas
except ImportError:
    %pip install -q git+https://gitlab.com/ivanvladimir/maquinas
    import maquinas

# Load machines

In [ ]:
from maquinas.io import load_fa, load_pda, load_tm
from maquinas.simulation import Simulation
import ipywidgets as widgets

In [ ]:
machines=[]

# Loading machines from strings
machines.append(load_fa("""---
label: "DFA: b in pairs"
description: The number of b symbols is even (DFA)
accepted: [ε, aabb, abab, abbaabbaaaaa]
rejected: [b, ab, bbb, aaaabaababaa]
---
        |  a  |  b  |
 ->q_0] | q_0 | q_1 |
   q_1  | q_1 | q_0 |
"""))

machines.append(load_fa("""---
label: "NDFA: contains ab"
description: The strings contain the ab substring (NDFA)
accepted: [ab, aabb, abab, aaaaaaaaaab, abbaabbaaaaa]
rejected: [ε, a, b, bbb, bbbbbaaaaaaaaaa]
---
        |  a       |  b  |
 ->q_0  | q_0, q_1 | q_0 |
   q_1  | q_0      | q_2 |
   q_2] | q_2      | q_2 |
"""))

machines.append(load_fa("""
---
label: "NDFA-ε: a follow by a b"
description: Aes or not aes ends with a b (NDFA-e)
accepted: [b, ab, aab, aaaaaab]
rejected: [ε, a, ba, abab, aaaaaaba]
---
        |  a       |  b  | epsilon |
 ->q_0  | q_1      |     |  q_1    |
   q_1  | q_1      | q_2 |         |  
   q_2] |          |     |         |
"""
))

machines.append(load_pda("""
---
label: "PDA: aⁿbⁿ"
description: A number of a's followed by the same number of b's (PDA)
accepted: [ε, ab, aabb, aaaaaaaabbbbbbbb]
rejected: [a, b, ba, aaabb, aabbbbb]
---
        |     ε     |          a          |    b    |
 ->q_0  | Z₀/Z₀→q_2 | Z₀/AZ₀→q_0,A/AA→q_0 | A/ε→q_1 |
   q_1  | Z₀/Z₀→q_2 |                     | A/ε→q_1 |
   q_2] |           |                     |         |
"""
))


machines.append(load_pda("""
---
label: "TSPDA: aⁿbⁿcⁿ"
description: A number of a's followed by the same number of b's, follow by the same number of c's (TSPDA)
accepted: [ε, abc, aabbcc, aaaaaaaabbbbbbbbcccccccc]
rejected: [a, b,c, ab, ba, ac, ca, cb, aaabbbcc, aabbccccc]
---
        |        ε        |                a                 |      b      |       c       |
 ->q_0  |                 | Z₀/AZ₀,Z₀/AZ₀→q_0, A/AA,A/AA→q_0 | A/ε,A/A→q_1 |               |
   q_1  |                 |                                  | A/ε,A/A→q_1 | Z₀/Z₀,A/ε→q_2 |
   q_2  | Z₀/Z₀,Z₀/Z₀→q_3 |                                  |             | Z₀/Z₀,A/ε→q_2 |
   q_3] |                 |                                  |             |               |
"""))

machines.append(load_tm("""
---
label: "LBA: aⁿbⁿ, n>0"
description: A number of a's followed by the same number of b's (LBA)
accepted: [ab, aabb, aaaaaaaabbbbbbbb]
rejected: [ε, a, b, ba, aaabb, aabbbbb]
---
        |     𝖁     |     X     |     Y     |     a     |     b     |
 ->q_0  |           |           | Y/Y→q_3:R | a/X→q_1:R |           |
   q_1  |           |           | Y/Y→q_1:R | a/a→q_1:R | b/Y→q_2:L |
   q_2  |           | X/X→q_0:R | Y/Y→q_2:L | a/a→q_2:L |           |
   q_3  | 𝖁/𝖁→q_4:R |           | Y/Y→q_3:R |           |           |
   q_4] |           |           |           |           |           |
""",lba=True))

machines.append(load_tm("""
---
label: "TM: aⁿbⁿ, n>0"
description: A number of a's followed by the same number of b's (MT)
accepted: [ab, aabb, aaaaaaaabbbbbbbb]
rejected: [ε, a, b, ba, aaabb, aabbbbb]
---
        |     𝖁     |     X     |     Y     |     a     |     b     |
 ->q_0  |           |           | Y/Y→q_3:R | a/X→q_1:R |           |
   q_1  |           |           | Y/Y→q_1:R | a/a→q_1:R | b/Y→q_2:L |
   q_2  |           | X/X→q_0:R | Y/Y→q_2:L | a/a→q_2:L |           |
   q_3  | 𝖁/𝖁→q_4:R |           | Y/Y→q_3:R |           |           |
   q_4] |           |           |           |           |           |
"""))

## Select the machine to explore

In [ ]:
dropdown = widgets.Dropdown(
    options=[(m.metadata.extra()['label'],m) for m in machines],
    description="Autómata:",
)
dropdown

In [ ]:
machine=dropdown.value

# Testing shared api with machines

## Prints elements of machine

Shows the information from the machine, short and long version

In [ ]:
machine.print_info()

In [ ]:
machine.print_long_info()

In [ ]:
machine.print_summary()

## Renders mathematical representation

In [ ]:
machine

In [ ]:
print(machine.latex())

Copy string and paste it in latex code, to text it here: https://latexeditor.lagrida.com/

## Shows transition function as table

In [ ]:
machine.table()

## Shows diagram of machine

In [ ]:
machine.draw()

## Testing string

In [ ]:
testing_string=machine.metadata.accepted[-1]
print("String to test:",testing_string)

In [ ]:
machine.accepts(testing_string)

In [ ]:
machine.print_transitions(testing_string)

In [ ]:
# recover last state
res=machine.delta_extended(None,testing_string)
res

In [ ]:
print(f"With state {res} can we accept?", "yes" if machine.acceptor(res) else "no" )

In [ ]:
print("Analysing string that should be accepted")
for t in machine.metadata.accepted:
    print(f"  Does {t if len(t) > 0 else 'ε' } is accepted?", "✓" if machine.accepts(t) else "𐄂" )

In [ ]:
print("Analysing string that should be rejected")
for t in machine.metadata.rejected:
    print(f"  Does {t if len(t) > 0 else 'ε' } is accepted?", "✓" if machine.accepts(t) else "𐄂" )

## Creating animation and showing it

In [ ]:
# Creates an animation and shows it
machine.save_animation(testing_string, "machine_animation.svg",show=True)

In [ ]:
machine.save_animation(machine.metadata.rejected[-1], "machine_animation.svg",show=True)

## Simulation

In [ ]:
s=Simulation(machine,testing_string)
s.run()